In [9]:
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas tqdm > /dev/null

import time
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# ============================
# CONFIGURAÇÕES DO SELENIUM
# ============================
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

# ============================
# CONFIGURAÇÕES
# ============================
base_url = "https://www.promofarma.com.br"
categoria = "medicamentos"   # 🔹 Troque a categoria aqui
max_paginas = 30             # limite de páginas para varrer (ajuste se precisar)

todos_produtos = []

# ============================
# LOOP EM TODAS AS PÁGINAS
# ============================
for page in range(1, max_paginas+1):
    url = f"{base_url}/{categoria}?page={page}"
    driver.get(url)
    time.sleep(2)

    # rolar até o fim para garantir que tudo carregue
    ultimo_altura = 0
    while True:
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        time.sleep(1)
        altura_atual = driver.execute_script("return document.body.scrollHeight")
        if altura_atual == ultimo_altura:
            break
        ultimo_altura = altura_atual

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # pegar só os blocos de produto
    cards = soup.select("div.product-card a")  # seletor mais específico
    if not cards:  # se não tiver mais produtos, para o loop
        break

    for c in cards:
        link = c.get("href")
        nome = c.get_text(strip=True)
        if not link or not nome:
            continue
        link = link if link.startswith("http") else base_url + link
        todos_produtos.append({
            "nome": nome,
            "link": link,
            "pagina": page
        })

    print(f"📦 Página {page}: {len(cards)} produtos")

driver.quit()

# ============================
# SALVAR RESULTADO
# ============================
df = pd.DataFrame(todos_produtos).drop_duplicates("link")
df.to_csv("produtos_promofarma.csv", index=False, encoding="utf-8-sig")
print(f"✅ Total final de produtos coletados: {len(df)}")
df.head()


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file


100%|██████████| 49/49 [00:00<00:00, 53354.33it/s]

✅ Total de produtos coletados: 43
📂 Arquivo produtos_promofarma.csv salvo!


,nome,link
0,"Pomada, Pasta e Gel Modelador",https://www.promofarma.com.br/beleza-e-higiene...
1,"Pomada, Pasta e Gel Modelador",https://www.promofarma.com.br/beleza-e-higiene...
2,Pré e Pós Barba,https://www.promofarma.com.br/beleza-e-higiene...
3,Protetor de Colchão,https://www.promofarma.com.br/beleza-e-higiene...
4,Pré e Pós Treino,https://www.promofarma.com.br/vitaminas-e-supl...
